# Lưu adapter bản cuối trên toàn pool (không GPU, không Internet)

In [ ]:
import hashlib, json, shutil
from pathlib import Path

ds = sorted(Path("/kaggle/input").rglob("cuoi_trang_thai.json"))
assert len(ds) == 1, ds
st = json.loads(ds[0].read_text(encoding="utf-8"))
assert st.get("xong") is True and not st.get("bat_thuong"), "ban cuoi chua xong"
OUT = Path("/kaggle/working/cuoi")
shutil.copytree(ds[0].parent, OUT)
for ten in ("cum_cuoi", "pair_cuoi"):
    v = st["viec"][ten]
    print(ten, "epoch", v["epoch_xong"], "so_title_bi_loai", v.get("so_title_bi_loai"), "kiem_nhanh chrF", v["kiem_nhanh"]["chrf"]["chrf_macro_huong"], flush=True)
    assert (OUT / ten / "ep2" / "adapter_model.safetensors").exists() and (OUT / ten / "ep2" / "adapter_config.json").exists(), ten
mf = {}
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        h = hashlib.sha256()
        with open(p, "rb") as a:
            for k in iter(lambda: a.read(2**20), b""):
                h.update(k)
        mf[str(p.relative_to(OUT)).replace("\\", "/")] = {"byte": p.stat().st_size, "sha256": h.hexdigest()}
Path("/kaggle/working/MANIFEST_CUOI.json").write_text(json.dumps(mf, indent=1), encoding="utf-8")
print(len(mf), "file", round(sum(v["byte"] for v in mf.values()) / 2**30, 2), "GiB", flush=True)
print("XONG", flush=True)
